# Chapter 5: Getting Started with pandas

## 5.2 Essential Functionality

### Reindexing

In [117]:
obj = pd.Series([4.5, 7.2, -5.3, 3.6], index=['d', 'b', 'a', 'c'])
obj

d    4.5
b    7.2
a   -5.3
c    3.6
dtype: float64

- Calling reindex on this Series **rearranges the data** according to the new index, **introducing missing values** if any index values were **not already present**:

In [118]:
obj2 = obj.reindex(['a', 'b', 'c', 'd', 'e'])
obj2

a   -5.3
b    7.2
c    3.6
d    4.5
e    NaN
dtype: float64

- For ordered data like time series, it may be desirable to do some **interpolation or filling of values** when reindexing. The method option allows us to do this, using a method such as ***ffill***, which **forward-fills the values**:

In [2]:
obj3 = pd.Series(['blue', 'purple', 'yellow'], index=[0, 2, 4])
obj3

0      blue
2    purple
4    yellow
dtype: object

In [3]:
obj3.reindex(range(6), method='ffill')

0      blue
1      blue
2    purple
3    purple
4    yellow
5    yellow
dtype: object

#### **예시**) 행 추가 + 행 순서 변경 + 열 추가 및 선택 

In [44]:
frame = pd.DataFrame(np.arange(9).reshape((3,3)),
                     index=['a', 'c', 'd'],
                     columns=['Ohio', 'Texas', 'California'])

states = ['Texas', 'Utah', 'California']

frame

,Ohio,Texas,California
a,0,1,2
c,3,4,5
d,6,7,8


In [45]:
# 방법_1 : reindex 사용
frame2 = frame.reindex(['a', 'b', 'c', 'd']) # b행 추가 및 행순서 변경
frame2

,Ohio,Texas,California
a,0.0,1.0,2.0
b,NaN,NaN,NaN
c,3.0,4.0,5.0
d,6.0,7.0,8.0


In [46]:
frame3 = frame2.reindex(columns = states) # Utah열 추가 Ohio 열제거, 열순서변경
frame3

,Texas,Utah,California
a,1.0,NaN,2.0
b,NaN,NaN,NaN
c,4.0,NaN,5.0
d,7.0,NaN,8.0


In [47]:
# 방법_2 : label-indexing with loc
frame.loc[['a', 'b', 'c', 'd'], states] #교재에서는 가능

KeyError: "['b'] not in index"

### Droping Entries from an Axis

- **Calling drop with a sequence of labels will drop values from the row labels (axis 0)**:
- Drop values from the columns by passing **axis=1 or axis='columns'**:
- Many functions, like drop, which modify the size or shape of a Series or DataFrame, can **manipulate an object *in-place* without returning a new object**:

In [17]:
data = pd.DataFrame(np.arange(16).reshape((4, 4)),
                    index=['Ohio', 'Colorado', 'Utah', 'New York'],
                    columns=['one', 'two', 'three', 'four'])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [10]:
# 행 drop
data.drop(['Colorado', 'Ohio']) #data 변경은 없음-> 새변수 할당 필요

,one,two,three,four
Utah,8,9,10,11
New York,12,13,14,15


In [11]:
# 열 drop
data.drop(['two', 'four'], axis=1) #or axis='columns'

,one,three
Ohio,0,2
Colorado,4,6
Utah,8,10
New York,12,14


In [18]:
# inplace 조건
data.drop(['New York'], inplace=True)
data #원 데이터프레임 자체가 변경, 주의 필요

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11


### Indexing, Selection, and Filtering

- Slicing with labels behaves differently than normal Python slicing in that **the end‐point is inclusive**

### Series (기본 행 접근)

In [43]:
obj = pd.Series(np.arange(4.), index=['a', 'b', 'c', 'd'])
obj

a    0.0
b    1.0
c    2.0
d    3.0
dtype: float64

- #### 단독 색인

```python
obj['b'] = obj[1] #레이블, 정수 모두 가능
```
- #### Fancy 색인

```python
obj[['b', 'd']] = obj[[1, 3]] #레이블, 정수 모두 가능
```
- #### Slicing 색인

```python
obj['b' : 'c'] = obj[1 : 3] #레이블로 슬라이싱 시 end도 포함
```

### DataFrame (기본 열 접근)

In [50]:
data = pd.DataFrame(np.arange(16).reshape((4,4)),
                   index=['Ohio', 'Colorado', 'Utah', 'New York'],
                   columns=['one', 'two', 'three', 'four'])
data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


- #### 단독 색인
```python
data['two'] #-> 열 접근, 레이블만 가능
```
> 단독 색인으로 레이블+정수 모두 행 접근 불가함

- #### Fancy 색인
```python
data[['three', 'one']] #-> 열 접근, 레이블만 가능
```
> Fancy 색인으로 레이블+정수 모두 행 접근 불가함

- #### Silicing 색인 -> 행 접근
```python
data['Utah':'New York'] = data[2:4] #-> 행 접근
```
> Slicing 색인으로 레이블+정수 모두 열 접근 불가함

- #### 행에 대한 접근을 위해 loc(레이블), iloc(정수) 도입
```python
data.loc['Colorado', ['two','three']]
data.iloc[1, [1, 2]]
data.iloc[2] #-> 단독 행 접근
```
> loc와 iloc에 대한 슬라이승 기법 사용 가능
>```python
data.loc[:'Utah', 'two']
Ohio			1
Colorado		5
Utah			9
#-------------------------------------
data.iloc[:, :3][data.three > 5]
			One		two		three
Colorado    4         5         6	
Utah		8         9         10
New York	12		13		14
```

### Integer Indexes (주의)
- To keep things consistent, if you have an **axis index containing integer**, data selection will always be **label-oriented**
- For more precise handling, use ***loc(레이블)*** or ***iloc(정수)***

In [3]:
ser = pd.Series(np.arange(3.))
ser

0    0.0
1    1.0
2    2.0
dtype: float64

In [7]:
ser[:1] #-> 정수로 인정하여 1 미포함

0    0.0
dtype: float64

In [8]:
ser.loc[:1] #-> 레이블로 인정하여 1포함

0    0.0
1    1.0
dtype: float64

In [9]:
ser.iloc[:1] #-> 정수로 인정하여 1마포함

0    0.0
dtype: float64